In [1]:
from pathlib import Path

METEO_CSV = Path("data") / "meteo" / "paris_montsouris_horaire.csv"

import requests
import pandas as pd

METEO_CSV.parent.mkdir(parents=True, exist_ok=True)

# ── Téléchargement via Open-Meteo (gratuit, sans auth) ──────────────────────
# Coordonnées GPS de la station Paris-Montsouris
LAT, LON = 48.8214, 2.3378

params = {
    "latitude"       : LAT,
    "longitude"      : LON,
    "start_date"     : "2022-01-01",
    "end_date"       : "2023-12-31",
    "hourly"         : "temperature_2m,precipitation,windspeed_10m,relativehumidity_2m,surface_pressure",
    "timezone"       : "Europe/Paris",
    "wind_speed_unit": "ms",
}

print("Téléchargement des données météo Paris-Montsouris 2022-2023...")
resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)
resp.raise_for_status()
data = resp.json()

# Aperçu de la structure retournée
print(f"  Clés disponibles : {list(data.keys())}")
print(f"  Variables horaires : {list(data['hourly'].keys())}")
print(f"  Nombre de points   : {len(data['hourly']['time'])}")

# Conversion en DataFrame et sauvegarde
df_meteo = pd.DataFrame(data["hourly"])
df_meteo.rename(columns={
    "time"               : "horodatage",
    "temperature_2m"     : "temperature",
    "precipitation"      : "precipitations",
    "windspeed_10m"      : "vent_ms",
    "relativehumidity_2m": "humidite",
    "surface_pressure"   : "pression",
}, inplace=True)

df_meteo.to_csv(METEO_CSV, index=False, sep=";")
print(f"  Sauvegardé : {METEO_CSV}")
print(f"  Lignes     : {len(df_meteo):,}")
df_meteo.head(3)

Téléchargement des données météo Paris-Montsouris 2022-2023...
  Clés disponibles : ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
  Variables horaires : ['time', 'temperature_2m', 'precipitation', 'windspeed_10m', 'relativehumidity_2m', 'surface_pressure']
  Nombre de points   : 17520
  Sauvegardé : data/meteo/paris_montsouris_horaire.csv
  Lignes     : 17,520


,horodatage,temperature,precipitations,vent_ms,humidite,pression
0,2022-01-01T00:00,9.4,0.0,3.22,94,1016.9
1,2022-01-01T01:00,9.4,0.0,3.11,94,1016.7
2,2022-01-01T02:00,10.7,0.0,2.50,95,1017.1
